In [ ]:
import pandas as pd

crop = "무"  # 작물 이름 변경
df = pd.read_csv(f"../data/{crop}_lag_인코딩스케일링완료.csv", encoding='cp949')

In [ ]:
# 결측치 확인
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

# 출력
if missing_cols.empty:
    print("결측치 없음!")
else:
    print("결측치 있는 컬럼:")
    print(missing_cols)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df['평균단가(원)'], bins=50, edgecolor='k')
plt.title("Target Distribution: 평균단가(원)")
plt.xlabel("가격")
plt.ylabel("빈도수")
plt.grid(True)
plt.show()


In [ ]:
correlations = df.corr(numeric_only=True)['평균단가(원)'].sort_values(ascending=False)
print("📊 평균단가와 상관관계가 높은 변수 Top 10:")
print(correlations.drop('평균단가(원)').head(10))


## Modeling

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import matplotlib.pyplot as plt

In [ ]:
X = df.drop(columns=['평균단가(원)', 'year'])
y = df['평균단가(원)']

# 시계열 정렬
X['week_start'] = pd.to_datetime(X['week_start'])
X = X.sort_values('week_start').reset_index(drop=True)
y = y.loc[X.index].reset_index(drop=True)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
rmse_list = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx].drop(columns=['week_start']), X.iloc[val_idx].drop(columns=['week_start'])
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # 모델 정의 및 학습
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)

    # 예측 및 평가
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_list.append(rmse)

    print(f"Fold {fold + 1} RMSE: {rmse:.2f}")
print("\n📊 전체 평가 결과")
print(f"평균 RMSE: {np.mean(rmse_list):.2f}")
print(f"최소 RMSE: {np.min(rmse_list):.2f}")
print(f"최대 RMSE: {np.max(rmse_list):.2f}")